<a href="https://colab.research.google.com/github/rayaan27/Fly-Rank-Internship/blob/main/notebooks/w01_research_question_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rayaan27/Fly-Rank-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Structured Content Archetype Clustering.**

I'm picking this lane because the starter dataset is exactly the shape it needs: 30,000 pages
across 32 clients, described entirely by numbers and categories (traffic, CTR, position, word
count, freshness, tiers) with no article text. That's a constraint, not a workaround — it means
I can honestly cluster on *behavior* (how a page performs) rather than pretend I'm clustering on
*meaning* (what a page is about), which this data can't support. An editor looking at a list of
30,000 pages one at a time can't hold that many patterns in their head; grouping pages that behave
alike turns an unmanageable list into a handful of archetypes they can actually act on.

In [1]:
import pandas as pd
import os

# Load the starter dataset. Try the repo-relative path first (this notebook lives in
# work/notebooks/, the data lives in data/raw/); fall back to the repo's raw file for
# standalone Colab runs.
LOCAL_PATH = "../../data/raw/content_refresh_anonymized.csv"
RAW_URL = "https://raw.githubusercontent.com/rayaan27/Fly-Rank-Internship/main/data/raw/content_refresh_anonymized.csv"

path = LOCAL_PATH if os.path.exists(LOCAL_PATH) else RAW_URL
df = pd.read_csv(path)

print(f"Loaded from: {'local repo path' if path == LOCAL_PATH else 'raw file (standalone run)'}")
print(f"Rows x columns: {df.shape}")
print(f"Distinct clients: {df['client_id'].nunique()}")
print(f"Content types: {df['content_type'].value_counts().to_dict()}")

# Confirm the lane's central constraint in code, not just in words: no article text or
# embedding column exists anywhere in the schema. This is the receipts for the "no semantic
# clustering" claim in Section 4.
text_like_cols = [c for c in df.columns if any(k in c.lower() for k in ["text", "body", "content_html", "embedding"])]
print(f"Text/embedding-like columns present: {text_like_cols if text_like_cols else 'none'}")

Loaded from: raw file (standalone run)
Rows x columns: (30000, 44)
Distinct clients: 32
Content types: {'keyword article': 27207, 'feedly article': 2096, 'comparison article': 697}
Text/embedding-like columns present: none


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

For a content editor deciding **where to spend limited rewrite/refresh time this quarter**, we
will build a set of **cluster profiles and archetype labels** from the trailing-90-day
performance and metadata in this inventory, describing which numeric/tier patterns
(traffic, CTR, position, freshness, word count) travel together — there is no target to
predict, so success is judged by cluster separation (e.g. silhouette score) plus whether a
human editor looks at a cluster's typical numbers and agrees it's a real, actionable group.

The editor acts directly on the archetype name: a page tagged **"stale visible"** gets queued
for a refresh; a page tagged **"weak/no-demand"** gets queued for pruning or merging; a
**"champion"** gets left alone (protect, don't touch). A wrong grouping costs real hours —
an editor could spend a day rewriting a page that was never going to move (wasted effort), or
ignore a page that was quietly cannibalizing a stronger sibling page (a missed opportunity
cost, not a catastrophic one, since nothing is deployed automatically from this). A plain
rule (e.g. "if impressions < 100, prune") isn't enough because pages behave differently along
*several* correlated axes at once — traffic, position, engagement, freshness — and no single
threshold captures a "type" of page the way a multi-signal cluster can.

In [2]:
# Back the "several correlated axes, not one threshold" claim with real numbers:
# show that the behavior signals don't move together, so a single if-statement can't
# capture the archetypes a multi-feature cluster could.

behavior_cols = ["impression_tier", "position_tier", "engagement_rate", "content_age_days"]
corr_cols = ["ctr", "avg_position", "engagement_rate", "content_age_days", "word_count"]

print("Correlation between candidate signals (not redundant with each other):")
print(df[corr_cols].corr(numeric_only=True).round(2))
print()

# A single "traffic" rule would miss pages that are visible but not engaging, or engaging
# but not visible -- both are real, different problems an editor should treat differently.
visible_low_engagement = ((df["impression_tier"].isin(["good", "excellent"])) & (df["engagement_rate"] < 1)).sum()
low_visible_high_engagement = ((df["impression_tier"] == "low") & (df["engagement_rate"] > df["engagement_rate"].median())).sum()
print(f"Visible but low-engagement pages (would be missed by a traffic-only rule): {visible_low_engagement}")
print(f"Low-visibility but above-median engagement pages (a different problem than 'no traffic'): {low_visible_high_engagement}")

Correlation between candidate signals (not redundant with each other):
                   ctr  avg_position  engagement_rate  content_age_days  \
ctr               1.00         -0.07             0.10              0.01   
avg_position     -0.07          1.00            -0.02              0.16   
engagement_rate   0.10         -0.02             1.00              0.04   
content_age_days  0.01          0.16             0.04              1.00   
word_count       -0.12          0.12            -0.06             -0.12   

                  word_count  
ctr                    -0.12  
avg_position            0.12  
engagement_rate        -0.06  
content_age_days       -0.12  
word_count              1.00  

Visible but low-engagement pages (would be missed by a traffic-only rule): 3283
Low-visibility but above-median engagement pages (a different problem than 'no traffic'): 679


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers from the actual inventory that argue for clustering over a single rule:

1. **Missingness tracks content type, not randomness**: `word_count` is missing for
   about 28% of "keyword article" rows but 0% of the other two content types — any feature
   set has to be built type-aware, which is exactly the kind of multi-column pattern
   clustering is built to organize.
2. **Performance is lopsided, not smooth**: `impression_tier` splits roughly
   11,200 low / 10,500 moderate / 7,200 good / 1,100 excellent — most of the inventory
   is quietly underperforming while a small slice is carrying the visibility. That's a
   strong hint that distinct groups exist rather than one continuous gradient.
3. **Rough archetype previews already show up in raw counts**: about 8.5% of pages look
   like weak/no-demand candidates (very low impressions and search volume), about 5.3%
   look like stale-visible candidates (page is over a year old but still classed
   "good"/"excellent" visibility), and about 10.9% are visible but barely engaging
   (an engagement-problem shape) — these numbers come from simple filters, not a model,
   but they show the archetypes this lane is chasing are already visible in the raw data.

In [3]:
# 1. Missingness tracks content_type (not random) -- the "gotcha" that shapes feature engineering
missing_by_type = df.groupby("content_type")["word_count"].apply(lambda s: round(s.isna().mean() * 100, 1))
print("word_count missing % by content_type:")
print(missing_by_type)
print()

# 2. Performance is lopsided across impression tiers
print("impression_tier counts (out of 30,000 pages):")
print(df["impression_tier"].value_counts())
print()

# 3. Rough, filter-based archetype previews (NOT a real clustering result -- just a sanity
# check that these shapes exist in the raw numbers before we commit to modeling them)
weak_demand = ((df["impressions_90d"] < 100) & (df["search_volume"] < 10)).sum()
stale_visible = ((df["content_age_days"] > 365) & (df["impression_tier"].isin(["good", "excellent"]))).sum()
engagement_problem = ((df["impression_tier"].isin(["good", "excellent"])) & (df["engagement_rate"] < 1)).sum()

n = len(df)
print(f"Weak/no-demand candidates:    {weak_demand:>5} ({weak_demand/n*100:.1f}%)")
print(f"Stale-visible candidates:     {stale_visible:>5} ({stale_visible/n*100:.1f}%)")
print(f"Engagement-problem candidates:{engagement_problem:>5} ({engagement_problem/n*100:.1f}%)")

word_count missing % by content_type:
content_type
comparison article     0.0
feedly article         0.0
keyword article       28.3
Name: word_count, dtype: float64

impression_tier counts (out of 30,000 pages):
impression_tier
low          11248
moderate     10469
good          7205
excellent     1078
Name: count, dtype: int64

Weak/no-demand candidates:     2547 (8.5%)
Stale-visible candidates:      1600 (5.3%)
Engagement-problem candidates: 3283 (10.9%)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this work CAN say:** which pages have *observed* similar behavior across traffic,
position, freshness, and engagement over the trailing 90 days; a *directional* read on which
group a given page currently sits in (e.g. "this page's numbers currently look like the
stale-visible group"); and *decision-support* suggestions such as protect, improve, rewrite,
merge, prune, or monitor, meant to help an editor prioritize, not to auto-execute.

**What this work will NEVER say:** that a cluster *causes* a page's performance, or that
being grouped with "champions" *will* make a page succeed — clusters describe a pattern in
the last 90 days, not a mechanism. It also will not claim to predict what Google's search
ranking algorithm will do next, and it will not claim to understand *what a page is about*.
Because this dataset has no article text, only metrics, buckets, token counts, and metadata,
the clustering here is strictly **metric clustering** — grouping by how pages behave — and I
will not call it "semantic clustering," which would mean grouping by what pages mean, a claim
this data cannot support. Cluster names are a lens for a human to sense-check, not a ground
truth label: I'll inspect what's actually inside a cluster before naming it, not the other
way around.

In [4]:
# Confirm in code what the claims above depend on: this dataset is metric/metadata only.
# No free-text article body, no embeddings -- so "semantic clustering" is not on the table.
print("All columns in the starter dataset:")
print(list(df.columns))
print()
print("Columns that could carry article MEANING (text, body, embeddings): ", end="")
semantic_capable = [c for c in df.columns if any(k in c.lower() for k in ["text", "body", "embedding", "vector"])]
print(semantic_capable if semantic_capable else "none -- confirms metric-only clustering, not semantic clustering")

All columns in the starter dataset:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Columns that could carry article MEANING (text, body, embeddings): none -- confirms metric-only clustering, not semantic clustering


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.